# 🚨 Water Quality Drift Detection & Analysis

## Purpose  
Advanced drift detection system monitoring water quality model performance and data patterns

## Detection Capabilities
1. **Feature Drift**: Statistical tests for all water quality parameters
2. **Prediction Drift**: Model output distribution analysis
3. **Label Drift**: Ground truth distribution changes (when available)
4. **Performance Degradation**: Real-world accuracy tracking

## Alert Framework
- 🟢 **Normal**: All metrics within thresholds
- 🟡 **Warning**: Statistical significance detected  
- 🔴 **Critical**: Major drift requiring immediate action
- 📧 **Notifications**: Automated alerts to operations team

## Techniques
- Statistical tests (KS, Jensen-Shannon)
- Population Stability Index (PSI)
- Performance monitoring
- Alert automation with actionable insights

In [ ]:
# 📦 Install required packages
%pip install pandas numpy scipy databricks-sdk scikit-learn plotly

# Restart Python to load new packages
dbutils.library.restartPython()

In [ ]:
# 📚 Import Libraries
import pandas as pd
import numpy as np
import json
from datetime import datetime, timedelta
from scipy import stats
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from databricks.sdk import WorkspaceClient
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count

# Initialize connections
spark = SparkSession.builder.getOrCreate()
w = WorkspaceClient()

print("✅ Libraries loaded for drift detection")

In [ ]:
# 🎯 Configuration - Environment Parameters
catalog_name = spark.conf.get("catalog_name", "dev_digital_engineering_services")  
schema_name = spark.conf.get("schema_name", "default")
alert_thresholds = {
    "feature_drift_warning": 0.05,    # p-value threshold for warning
    "feature_drift_critical": 0.01,    # p-value threshold for critical
    "psi_warning": 0.1,               # PSI threshold for warning  
    "psi_critical": 0.25,             # PSI threshold for critical
    "performance_degradation": 0.05    # Accuracy drop threshold
}

# Table names
INFERENCE_TABLE = f"{catalog_name}.{schema_name}.water_quality_inferences"
TRAINING_TABLE = f"{catalog_name}.{schema_name}.water_quality_features"

# Water quality features to monitor
WATER_FEATURES = ['ph', 'Hardness', 'Solids', 'Chloramines', 'Sulfate', 
                  'Conductivity', 'Organic_carbon', 'Trihalomethanes', 'Turbidity']

ENGINEERED_FEATURES = ['ph_normalized', 'hardness_ratio', 'organic_load', 'water_quality_index']

print(f"🚨 Drift Detection Configuration:")
print(f"   📊 Inference Table: {INFERENCE_TABLE}")
print(f"   📈 Training Table: {TRAINING_TABLE}")
print(f"   🎯 Features Monitored: {len(WATER_FEATURES + ENGINEERED_FEATURES)}")
print(f"   ⚠️ Warning PSI: {alert_thresholds['psi_warning']}")
print(f"   🔴 Critical PSI: {alert_thresholds['psi_critical']}")

In [ ]:
# 📊 Load Reference and Current Data
print("📊 Loading baseline and current data for drift analysis...")

# Load training data as baseline reference
baseline_data = spark.sql(f"""
    SELECT * FROM {TRAINING_TABLE} 
    WHERE dataset_split = 'train'
    ORDER BY RAND()
    LIMIT 1000
""").toPandas()

# Load recent inference data (last 7 days)
current_data = spark.sql(f"""
    SELECT * FROM {INFERENCE_TABLE}
    WHERE prediction_timestamp >= CURRENT_TIMESTAMP() - INTERVAL 7 DAYS
    ORDER BY prediction_timestamp DESC
""").toPandas()

print(f"✅ Data loaded:")
print(f"   📚 Baseline (Training): {len(baseline_data):,} records")
print(f"   🔄 Current (Inference): {len(current_data):,} records")

if len(current_data) == 0:
    print("❌ No recent inference data found")
    print("💡 Run batch inference job to generate data")
    raise Exception("No current data for drift analysis")

# Separate normal vs drift simulation data
normal_data = current_data[current_data.get('drift_simulation', False) == False]
drift_data = current_data[current_data.get('drift_simulation', False) == True]

print(f"   📊 Normal Inference: {len(normal_data):,} records")
print(f"   🌊 Drift Simulation: {len(drift_data):,} records")

In [ ]:
# 📈 Statistical Drift Tests
def calculate_psi(baseline, current, buckets=10):
    """Calculate Population Stability Index"""
    
    # Create bins based on baseline distribution
    bins = pd.qcut(baseline, buckets, duplicates='drop', retbins=True)[1]
    
    # Calculate distributions
    baseline_dist = pd.cut(baseline, bins).value_counts(normalize=True, sort=False)
    current_dist = pd.cut(current, bins).value_counts(normalize=True, sort=False)
    
    # Handle zeros to avoid division issues
    baseline_dist = np.where(baseline_dist == 0, 0.0001, baseline_dist)
    current_dist = np.where(current_dist == 0, 0.0001, current_dist)
    
    # Calculate PSI
    psi = np.sum((current_dist - baseline_dist) * np.log(current_dist / baseline_dist))
    
    return psi

def interpret_psi(psi_value):
    """Interpret PSI value"""
    if psi_value < 0.1:
        return "🟢 No Drift", "normal"
    elif psi_value < 0.25:
        return "🟡 Warning", "warning"
    else:
        return "🔴 Critical", "critical"

print("📈 Conducting statistical drift tests...")

# Results storage
drift_results = {}

# Test each water quality feature
for feature in WATER_FEATURES + ENGINEERED_FEATURES:
    if feature in baseline_data.columns and feature in current_data.columns:
        
        # Extract feature values
        baseline_feature = baseline_data[feature].dropna()
        current_feature = current_data[feature].dropna()
        
        if len(baseline_feature) > 0 and len(current_feature) > 0:
            # Kolmogorov-Smirnov test
            ks_stat, ks_p_value = stats.ks_2samp(baseline_feature, current_feature)
            
            # Population Stability Index  
            psi = calculate_psi(baseline_feature, current_feature)
            psi_status, psi_level = interpret_psi(psi)
            
            # Jensen-Shannon divergence (for distributions)
            try:
                # Create normalized histograms
                hist1, bins = np.histogram(baseline_feature, bins=20, density=True)
                hist2, _ = np.histogram(current_feature, bins=bins, density=True)
                
                # Normalize to probabilities
                hist1 = hist1 / np.sum(hist1)
                hist2 = hist2 / np.sum(hist2)
                
                # Calculate JS divergence
                m = 0.5 * (hist1 + hist2)
                js_div = 0.5 * stats.entropy(hist1, m) + 0.5 * stats.entropy(hist2, m)
            except:
                js_div = None
            
            # Store results
            drift_results[feature] = {
                'ks_statistic': ks_stat,
                'ks_p_value': ks_p_value,
                'psi': psi,
                'psi_status': psi_status,
                'psi_level': psi_level,
                'js_divergence': js_div,
                'baseline_mean': baseline_feature.mean(),
                'current_mean': current_feature.mean(),
                'mean_change_pct': ((current_feature.mean() - baseline_feature.mean()) / baseline_feature.mean()) * 100
            }

print(f"✅ Completed drift tests for {len(drift_results)} features")

In [ ]:
# 🎯 Analyze Results and Alert Generation
print("🎯 Analyzing drift detection results...")

# Categorize features by drift severity
critical_drift = []
warning_drift = []
normal_features = []

print(f"\n📊 DRIFT DETECTION RESULTS:")
print(f"{'Feature':<20} {'PSI':<8} {'Status':<15} {'KS p-val':<10} {'Mean Change':<12}")
print("-" * 75)

for feature, results in drift_results.items():
    psi = results['psi']
    status = results['psi_status']
    ks_p = results['ks_p_value']
    mean_change = results['mean_change_pct']
    
    print(f"{feature:<20} {psi:<8.3f} {status:<15} {ks_p:<10.3f} {mean_change:<12.1f}%")
    
    # Categorize by severity
    if results['psi_level'] == 'critical':
        critical_drift.append(feature)
    elif results['psi_level'] == 'warning':
        warning_drift.append(feature)
    else:
        normal_features.append(feature)

# Overall system status
print(f"\n🎯 SYSTEM DRIFT STATUS:")
print(f"   🔴 Critical Drift: {len(critical_drift)} features")
print(f"   🟡 Warning Level: {len(warning_drift)} features")  
print(f"   🟢 Normal: {len(normal_features)} features")

# Determine overall alert level
if len(critical_drift) > 0:
    overall_status = "🔴 CRITICAL"
    overall_level = "critical"
elif len(warning_drift) > 2:  # More than 2 features in warning
    overall_status = "🟡 WARNING"  
    overall_level = "warning"
else:
    overall_status = "🟢 NORMAL"
    overall_level = "normal"

print(f"\n🚨 OVERALL STATUS: {overall_status}")

In [ ]:
# 🔍 Performance Degradation Analysis
print("🔍 Analyzing model performance degradation...")

# Check if ground truth is available for recent predictions
has_ground_truth = 'Potability' in current_data.columns and current_data['Potability'].notna().any()

performance_analysis = {}

if has_ground_truth:
    print("✅ Ground truth available - analyzing performance metrics")
    
    # Calculate metrics for different time periods
    time_periods = {
        'last_24h': current_data[current_data['prediction_timestamp'] >= datetime.now() - timedelta(days=1)],
        'last_3d': current_data[current_data['prediction_timestamp'] >= datetime.now() - timedelta(days=3)],
        'last_7d': current_data
    }
    
    baseline_accuracy = 0.85  # Expected baseline accuracy from training
    
    for period_name, period_data in time_periods.items():
        if len(period_data) > 10:  # Need minimum samples
            y_true = period_data['Potability']
            y_pred = period_data['predicted_potability']
            
            # Calculate metrics
            accuracy = accuracy_score(y_true, y_pred)
            precision = precision_score(y_true, y_pred, zero_division=0)
            recall = recall_score(y_true, y_pred, zero_division=0)
            f1 = f1_score(y_true, y_pred, zero_division=0)
            
            # Calculate degradation
            accuracy_drop = baseline_accuracy - accuracy
            
            performance_analysis[period_name] = {
                'accuracy': accuracy,
                'precision': precision,
                'recall': recall,
                'f1': f1,
                'accuracy_drop': accuracy_drop,
                'sample_size': len(period_data),
                'degraded': accuracy_drop > alert_thresholds['performance_degradation']
            }
    
    # Show performance analysis
    print(f"\n📊 PERFORMANCE ANALYSIS:")
    print(f"{'Period':<10} {'Accuracy':<10} {'Precision':<10} {'Recall':<10} {'F1':<10} {'Drop':<8} {'Status'}")
    print("-" * 75)
    
    for period, metrics in performance_analysis.items():
        status = "🔴 DEGRADED" if metrics['degraded'] else "🟢 OK"
        print(f"{period:<10} {metrics['accuracy']:<10.3f} {metrics['precision']:<10.3f} "
              f"{metrics['recall']:<10.3f} {metrics['f1']:<10.3f} "
              f"{metrics['accuracy_drop']:<8.3f} {status}")
              
else:
    print("⚠️ No ground truth available - performance analysis skipped")
    print("💡 Consider collecting ground truth labels for production monitoring")

In [ ]:
# 🚨 Advanced Alert Generation  
print("🚨 Generating actionable alerts...")

alerts = []

# Feature drift alerts
if critical_drift:
    alerts.append({
        'level': 'critical',
        'type': 'feature_drift',
        'message': f"Critical drift detected in {len(critical_drift)} features: {', '.join(critical_drift)}",
        'features': critical_drift,
        'action': 'Immediate retraining recommended',
        'urgency': 'high'
    })

if warning_drift:
    alerts.append({
        'level': 'warning',
        'type': 'feature_drift', 
        'message': f"Warning-level drift in {len(warning_drift)} features: {', '.join(warning_drift)}",
        'features': warning_drift,
        'action': 'Monitor closely, consider retraining',
        'urgency': 'medium'
    })

# Performance degradation alerts
degraded_periods = [p for p, m in performance_analysis.items() if m.get('degraded', False)]
if degraded_periods:
    worst_period = max(degraded_periods, 
                      key=lambda p: performance_analysis[p]['accuracy_drop'])
    worst_drop = performance_analysis[worst_period]['accuracy_drop']
    
    alerts.append({
        'level': 'critical',
        'type': 'performance_degradation',
        'message': f"Model accuracy dropped by {worst_drop:.1%} in {worst_period}",
        'period': worst_period, 
        'accuracy_drop': worst_drop,
        'action': 'Investigate data quality and retrain model',
        'urgency': 'high'
    })

# Water quality specific alerts
water_quality_alerts = []
for feature in ['ph', 'Chloramines', 'Trihalomethanes']:  # Safety-critical features
    if feature in critical_drift:
        water_quality_alerts.append({
            'level': 'critical',
            'type': 'safety_parameter_drift',
            'message': f"Safety-critical parameter {feature} shows significant drift",
            'feature': feature,
            'action': 'Verify water treatment systems immediately',
            'urgency': 'highest'
        })

alerts.extend(water_quality_alerts)

# Display alerts
print(f"\n🚨 GENERATED ALERTS ({len(alerts)} total):")
if not alerts:
    print("✅ No alerts - system operating normally")
else:
    for i, alert in enumerate(alerts, 1):
        print(f"\n📢 Alert {i} [{alert['level'].upper()}]:")
        print(f"   📋 Type: {alert['type']}")
        print(f"   💬 Message: {alert['message']}")
        print(f"   🎯 Action: {alert['action']}")
        print(f"   ⚡ Urgency: {alert['urgency']}")

In [ ]:
# 📧 Alert Notification System  
print("📧 Processing alert notifications...")

# Create notification content
notification_content = {
    'timestamp': datetime.now().isoformat(),
    'overall_status': overall_level,
    'total_alerts': len(alerts),
    'critical_alerts': len([a for a in alerts if a['level'] == 'critical']),
    'warning_alerts': len([a for a in alerts if a['level'] == 'warning']),
    'drift_summary': {
        'critical_features': len(critical_drift),
        'warning_features': len(warning_drift),
        'normal_features': len(normal_features)
    },
    'performance_summary': performance_analysis,
    'detailed_alerts': alerts,
    'recommendations': []
}

# Generate recommendations based on drift patterns
if critical_drift:
    notification_content['recommendations'].append(
        "Immediate model retraining required due to critical feature drift"
    )
    
if 'ph' in critical_drift or 'Chloramines' in critical_drift:
    notification_content['recommendations'].append(
        "Water safety parameters drifted - verify treatment systems"
    )

if len(warning_drift) > 3:
    notification_content['recommendations'].append(
        "Multiple features showing drift - investigate data source changes"
    )

if not alerts:
    notification_content['recommendations'].append(
        "System operating normally - continue monitoring"
    )

# In production, this would send to:
# - Email/Slack notifications  
# - Dashboard updates
# - Ticketing system
# - MLOps platform alerts

print(f"✅ Notification content prepared:")
print(f"   📊 Overall Status: {overall_status}")
print(f"   🚨 Total Alerts: {len(alerts)}")
print(f"   💡 Recommendations: {len(notification_content['recommendations'])}")

# Save notification to file (in production: send via email/webhook)
import json
notification_json = json.dumps(notification_content, indent=2, default=str)

# Display key recommendations
print(f"\n💡 KEY RECOMMENDATIONS:")
for i, rec in enumerate(notification_content['recommendations'], 1):
    print(f"   {i}. {rec}")

In [ ]:
# 📊 Summary Report Generation
print("📊 Generating comprehensive drift detection report...")

# Create executive summary
executive_summary = f"""
🌊 WATER QUALITY MODEL DRIFT DETECTION REPORT
Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

EXECUTIVE SUMMARY:
• Overall System Status: {overall_status}
• Total Features Monitored: {len(WATER_FEATURES + ENGINEERED_FEATURES)}
• Data Period: Last 7 days ({len(current_data):,} inference records)
• Baseline Comparison: Training dataset ({len(baseline_data):,} records)

DRIFT DETECTION RESULTS:
• 🔴 Critical Drift: {len(critical_drift)} features
• 🟡 Warning Level: {len(warning_drift)} features  
• 🟢 Normal: {len(normal_features)} features

ALERTS GENERATED: {len(alerts)} total
• Critical: {len([a for a in alerts if a['level'] == 'critical'])}
• Warning: {len([a for a in alerts if a['level'] == 'warning'])}
"""

if has_ground_truth:
    executive_summary += f"""
PERFORMANCE MONITORING:
• Accuracy Baseline: {baseline_accuracy:.1%}
• Recent Performance: {performance_analysis.get('last_24h', {}).get('accuracy', 0):.1%}
"""

print(executive_summary)

# Detailed feature analysis
print(f"\n📈 DETAILED FEATURE ANALYSIS:")
print(f"Most Significant Drifts:")

# Sort by PSI score
sorted_features = sorted(drift_results.items(), 
                        key=lambda x: x[1]['psi'], reverse=True)

for feature, results in sorted_features[:5]:  # Top 5
    psi = results['psi']
    status = results['psi_status']
    mean_change = results['mean_change_pct']
    
    print(f"  • {feature}:")
    print(f"    PSI: {psi:.3f} ({status})")
    print(f"    Mean change: {mean_change:+.1f}%")
    print(f"    KS test p-value: {results['ks_p_value']:.3f}")

print(f"\n✅ DRIFT DETECTION COMPLETED!")
print(f"🎯 System Status: {overall_status}")
print(f"📊 Monitoring Dashboard: Check Unity Catalog Quality Monitors")
print(f"🔔 Notifications: {len(alerts)} alerts generated")

if alerts:
    print(f"\n⚡ IMMEDIATE ACTIONS REQUIRED:")
    urgent_alerts = [a for a in alerts if a.get('urgency') in ['high', 'highest']]
    for alert in urgent_alerts[:3]:  # Show top 3 urgent
        print(f"  • {alert['action']}")

print(f"\n🔄 NEXT STEPS:")
print("1. Review detailed results in monitoring dashboard") 
print("2. Investigate root causes of significant drifts")
print("3. Consider model retraining if critical drift detected")
print("4. Update alert thresholds based on business requirements")
print("5. Schedule regular drift detection runs (daily/weekly)")